# Repair arm — SAM 2.1 Hiera-L over the selected real capture frames

Design note: `docs/repair_arm_design_note.md`. This notebook performs ONLY
the model-inference stage: 16-32 real ARKitScenes RGB frames in, per-frame
class-agnostic masks out. Frame selection, lifting, association, fusion,
pooling and evaluation all run locally in the repo.

**Pins (do not edit):** sam2 @ `2b90b9f5ceec907a1c18123530e92e794ad901a4`,
checkpoint `sam2.1_hiera_large.pt` sha256
`2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318`,
automatic-mask-generator parameters exactly as in
`docs/c1_p1_multiview_proposals_protocol.md`, seeds 0. Same pin as C1-P1 —
the mask SOURCE is the variable under test, not the model configuration.

**Difference from `c1p1_sam2_colab.ipynb`:** that notebook segmented 40
point-splat renders of the mesh. This one segments the real photographs the
device took. Nothing else about the inference changes.

**Isolation:** upload ONLY `repair_frames_<scene_id>.tar.gz` from
`tools/arkitscenes_repair_frames.py --tar`. It contains RGB PNGs and an
upload manifest. Never upload poses, the mesh, id buffers, Mask3D output or
annotations.

**Budget:** ONE run per scene, 41069021 first. If 41069021 fails its gates
locally, do NOT run 41069025.

In [ ]:
# [1] environment + pinned checkpoint (verify sha BEFORE any inference)
import hashlib
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!git checkout 2b90b9f5ceec907a1c18123530e92e794ad901a4
!pip install -q -e .
!wget -q -O sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sha = hashlib.sha256(open('sam2.1_hiera_large.pt','rb').read()).hexdigest()
assert sha == '2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318', f'CHECKPOINT SHA MISMATCH: {sha}'
print('checkpoint sha OK:', sha)
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# [2] frames: upload repair_frames_<SCENE_ID>.tar.gz (PNGs + upload manifest)
SCENE_ID = 'arkitscenes_41069021'   # change ONLY per the staged protocol order

from google.colab import drive
drive.mount('/content/drive')
import tarfile, glob, json, os
tar = f'/content/drive/MyDrive/repair/repair_frames_{SCENE_ID}.tar.gz'
tarfile.open(tar).extractall('/content/frames')
root = f'/content/frames/repair_frames_{SCENE_ID}'
manifest = json.load(open(os.path.join(root, 'upload_manifest.json')))
pngs = [os.path.join(root, f['png']) for f in manifest['frames']]
assert len(pngs) == manifest['n_frames'], 'frame count disagrees with the manifest'
assert 16 <= len(pngs) <= 32, f'{len(pngs)} frames is outside the 16-32 band'

# Every image must match the sha256 the local selection recorded. A silently
# re-encoded or reordered upload would produce masks that lift onto the wrong
# geometry, and nothing downstream could detect it.
for f, p in zip(manifest['frames'], pngs):
    got = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    assert got == f['sha256'], f"{f['png']}: sha256 {got} != manifest {f['sha256']}"
SELECTION_SHA = manifest['selection_sha256']
print('frames ready:', len(pngs), '| selection', SELECTION_SHA[:16])

In [ ]:
# [3] pinned automatic-mask inference (seeds 0; one pass; no retries)
# Every parameter below is frozen. Do not edit anything here.
import random, time, numpy as np, torch
from PIL import Image
random.seed(0); np.random.seed(0); torch.manual_seed(0)
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

model = build_sam2('configs/sam2.1/sam2.1_hiera_l.yaml',
                   'sam2.1_hiera_large.pt', device='cuda')
gen = SAM2AutomaticMaskGenerator(
    model, points_per_side=32, points_per_batch=64,
    pred_iou_thresh=0.8, stability_score_thresh=0.95,
    stability_score_offset=1.0, mask_threshold=0.0, box_nms_thresh=0.7,
    crop_n_layers=0, crop_nms_thresh=0.7, min_mask_region_area=0,
    use_m2m=False, multimask_output=True, output_mode='uncompressed_rle')

def rle_to_mask(rle):
    h, w = rle['size']
    flat = np.zeros(h * w, dtype=np.uint8)
    vals = np.zeros(len(rle['counts']), dtype=np.uint8); vals[1::2] = 1
    flat[:] = np.repeat(vals, rle['counts'])
    return flat.reshape((h, w), order='F')

out, scores, shapes, t0 = {}, {}, {}, time.time()
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    for k, p in enumerate(pngs):
        img = np.array(Image.open(p).convert('RGB'))
        anns = gen.generate(img)
        h, w = img.shape[:2]
        # Masks are stored packed per frame. They OVERLAP and do NOT tile the
        # image; the local stage lifts each one independently and never forms
        # a complement. Do not post-process them into a partition here.
        packed = (np.stack([np.packbits(rle_to_mask(a['segmentation']).ravel())
                            for a in anns]) if anns
                  else np.zeros((0, (h * w + 7) // 8), np.uint8))
        out[f'masks_{k:02d}'] = packed
        scores[f'scores_{k:02d}'] = np.array(
            [[a['predicted_iou'], a['stability_score']] for a in anns],
            dtype=np.float32).reshape(-1, 2)
        shapes[f'shape_{k:02d}'] = np.array([h, w], dtype=np.int32)
        print(f'frame {k:02d}: {len(anns)} masks  ({time.time()-t0:.0f}s)')
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 2**30
print(f'done in {elapsed:.0f}s, peak {peak:.2f} GiB')

In [ ]:
# [4] save sidecar (packbits masks + scores + shapes + env) to Drive
import json, platform
env = dict(scene_id=SCENE_ID,
           selection_sha256=SELECTION_SHA,
           sam2_commit='2b90b9f5ceec907a1c18123530e92e794ad901a4',
           checkpoint_sha256=sha,
           torch=torch.__version__, cuda=torch.version.cuda,
           device=torch.cuda.get_device_name(0),
           python=platform.python_version(),
           elapsed_seconds=round(elapsed, 1), peak_vram_gib=round(peak, 2),
           n_frames=len(pngs), seeds=0,
           points_per_side=32, pred_iou_thresh=0.8,
           stability_score_thresh=0.95, box_nms_thresh=0.7,
           multimask_output=True, output_mode='uncompressed_rle',
           masks_are_overlapping_hypotheses=True)
dst = f'/content/drive/MyDrive/repair/repair_sam_masks_{SCENE_ID}.npz'
np.savez_compressed(dst, env=json.dumps(env), **out, **scores, **shapes)
print('saved:', dst)
print(json.dumps(env, indent=1))

## After this notebook (local, in the repo)

```
python3 tools/arkitscenes_repair_propose_sam.py --scene 41069021 \
    --masks runs/arkitscenes_repair/arkitscenes_41069021/repair_sam_masks_arkitscenes_41069021.npz
python3 tools/arkitscenes_repair_eval.py --scene 41069021 \
    --repair runs/arkitscenes_repair/arkitscenes_41069021/repair_bank_sam.npz
```

Gate failure on 41069021 = STOP. Do not run 41069025, and do not touch the
unseen scene 47331972.